In [1]:
import pandas as pd

clusters_df = pd.read_csv("../analysis/reviews_with_clusters.csv")
cluster_keywords_df = pd.read_csv("../analysis/cluster_keywords.csv")

print("Clusters DF shape:", clusters_df.shape)
print("Cluster Keywords DF shape:", cluster_keywords_df.shape)
print("Sample keywords:\n", cluster_keywords_df.head())

Clusters DF shape: (200000, 14)
Cluster Keywords DF shape: (50, 3)
Sample keywords:
    cluster                                           keywords      domain
0        0  price, amazon, cup, order, flavor, beans, tast...  human_food
1        1  flavor, taste, cup, strong, bold, roast, bitte...  human_food
2        2  dogs, loves, treat, training, small, little, e...    pet_food
3        3  taste, sweet, use, free, syrup, flavor, sugar ...  human_food
4        4  candy, box, gift, chocolate, bought, little, k...  human_food


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.retrieval import retrieve_reviews, get_cluster_distribution

# pick a sample query
query = "Top complaints about coffee taste?"
top_k = 5

results = retrieve_reviews(query, k=top_k)
print("Retrieved review dicts:")
for r in results:
    print(r)

# Test cluster distribution
clusters_df = pd.read_csv("../analysis/reviews_with_clusters.csv")
indices = [r["index"] for r in results]
cluster_info = get_cluster_distribution(indices, clusters_df)
print("Cluster distribution:", cluster_info)

C:\Users\Arushi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


USING FIXED PATH VERSION
INDEX PATH: c:\Users\Arushi\Desktop\DS_Projects\InsightLens\faiss_index\amazon_reviews_index.faiss
MAPPING PATH: c:\Users\Arushi\Desktop\DS_Projects\InsightLens\embeddings\review_id_mapping.json


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3967.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieved review dicts:
{'product_id': 'B001EQ4F14', 'category': 'food', 'rating': 2, 'review_text': "This is the first time I've ever reviewed any product anywhere. I was so disgusted by this coffee that I found it necessary to do so. It has a sharp metallic taste that I will avoid in the future. I recommend Nescafe.", 'df_index': 11473, 'index': 11473, 'score': 0.7648504972457886}
{'product_id': 'B004ZIER34', 'category': 'food', 'rating': 5, 'review_text': 'All coffee flavor , unlike the other weird tasting non-caffeinated coffees.<br />I hate the "acidic taste " that most others have.<br /><br />Try it and you\'ll see what you have been missing.', 'df_index': 55397, 'index': 55397, 'score': 0.7609409093856812}
{'product_id': 'B000HBMHVM', 'category': 'food', 'rating': 3, 'review_text': 'My first taste impression of the coffee is that it is not great.<br /><br />I like dark coffee and this is pretty mild.', 'df_index': 129344, 'index': 129344, 'score': 0.7593860626220703}
{'product_i

In [3]:
from src.theme_extraction import generate_insight

# Assume cluster_keywords_df is loaded
insight = generate_insight(
    [r["review_text"] for r in results], 
    cluster_info, 
    cluster_keywords_df
)
print("Generated insight:", insight)

Generated insight: {'dominant_cluster': 1, 'keywords': 'flavor, taste, cup, strong, bold, roast, bitter, flavored, dark, vanilla', 'insight_text': 'Dominant cluster 1 with keywords: flavor, taste, cup, strong, bold, roast, bitter, flavored, dark, vanilla'}


In [4]:
from src.summary import summarize_reviews

sample_reviews = [
    "The coffee was too bitter and the packaging was torn.",
    "Loved the aroma but the taste was inconsistent.",
    "Delivery was quick but the coffee lost flavor after a week."
]

summary = summarize_reviews(sample_reviews)
print(summary)

C:\Users\Arushi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Arushi\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 76

{'short_summary': 'Summarize the following reviews:\nThe coffee was too bitter and the packaging was torn. Loved the aroma but the taste was inconsistent. Delivery was quick but the coffee lost flavor after a week.\nSummary:\nThe coffee was too bitter and the packaging was torn. Loved the aroma but the taste was inconsistent. Delivery was quick but the coffee lost flavor after a week.\nThe coffee was too bitter and the packaging was torn. Loved the aroma but the taste was inconsistent. Delivery was quick but the coffee lost flavor after a week.\nThe coffee was too bitter and the packaging was torn. Loved the aroma but the taste was inconsistent. Delivery was quick but the coffee lost flavor after a week.\nThe coffee was too bitter and the packaging was torn. Loved the aroma but the taste was inconsistent. Delivery was quick but the coffee lost flavor after a week.\nThe coffee was too bitter and the packaging was torn. Loved the aroma but the taste was inconsistent. Delivery was quick b

In [7]:
from src.pipeline import full_pipeline

query = "Top complaints about coffee taste?"
output = full_pipeline(query, k=5)

# Inspect output
print("Pipeline output keys:", output.keys())
print("Top reviews:", output["top_reviews"][:3])
print("Cluster info:", output["cluster_info"])
print("Insight:", output["insight"])
print("Summary:", output["summary"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pipeline output keys: dict_keys(['query', 'top_reviews', 'cluster_info', 'insight', 'summary'])
Top reviews: ["This is the first time I've ever reviewed any product anywhere. I was so disgusted by this coffee that I found it necessary to do so. It has a sharp metallic taste that I will avoid in the future. I recommend Nescafe.", 'All coffee flavor , unlike the other weird tasting non-caffeinated coffees.<br />I hate the "acidic taste " that most others have.<br /><br />Try it and you\'ll see what you have been missing.', 'My first taste impression of the coffee is that it is not great.<br /><br />I like dark coffee and this is pretty mild.']
Cluster info: {'dominant_cluster': 1, 'cluster_counts': {11: 1, 1: 4}, 'top_clusters': [(1, 4), (11, 1)]}
Insight: {'dominant_cluster': 1, 'keywords': 'flavor, taste, cup, strong, bold, roast, bitter, flavored, dark, vanilla', 'insight_text': 'Dominant cluster 1 with keywords: flavor, taste, cup, strong, bold, roast, bitter, flavored, dark, vanilla